#Build Constructors Dimension 

1. Read silver `constructors` table
2. Read gold `ref_nationality_region` table
3. Join the data from `constructors` with `ref_nationality_region` using `nationality`
4. Select the required columns
    - constructors.constructor_id
    - constructors.constructor_name
    - constructors.nationality
    - ref_nationality_region.region
5. Write the transformed data to gold `dim_constructors` table


In [0]:
dbutils.widgets.text('p_batch_id', '')
v_batch_id = dbutils.widgets.get('p_batch_id')

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.gold-helpers

In [0]:
target_table = f'{catalog_name}.{gold_schema}.dim_constructors'

In [0]:
constructors_df = spark.read.table(f'{catalog_name}.{silver_schema}.constructors').filter((F.col('batch_id') == v_batch_id))
region_df = spark.read.table(f'{catalog_name}.{gold_schema}.ref_nationality_region')

In [0]:
dim_constructors = (
    constructors_df
        .join(
            region_df,
            constructors_df.nationality == region_df.nationality,
            'left'
        ).select(
            constructors_df.constructor_id,
            constructors_df.constructor_name,
            constructors_df.nationality,
            region_df.region
        )
)

In [0]:
write_to_gold (
    df = dim_constructors,
    target_table = target_table,
    table_key = 's.constructor_id == t.constructor_id',
    columns_to_update = [
        'constructor_name',
        'nationality',
        'region'
    ]
)

In [0]:
# (
#     dim_constructors.write
#         .format('delta')
#         .mode('overwrite')
#         .saveAsTable(target_table)
# )

In [0]:
%sql
SELECT * FROM formula1.gold.dim_constructors

In [0]:
%sql
SELECT region, COUNT(*) as constructors_per_region FROM formula1.gold.dim_constructors GROUP BY region

In [0]:
%sql
SELECT * FROM formula1.gold.dim_constructors WHERE region = 'South America'